# Custom Comment Prediction Notebook (Self-contained)

This notebook loads an already trained HF model and predicts sentiment
on custom comments without external scripts.


In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# ===== configuration =====
THEME = "all_sources"  # usually all_sources for global prediction

cwd = Path.cwd().resolve()
if (cwd / "requirements.txt").exists() and (cwd / "notebooks").exists():
    ROOT_DIR = cwd
elif (cwd.parent / "requirements.txt").exists() and (cwd.parent / "notebooks").exists():
    ROOT_DIR = cwd.parent
else:
    raise FileNotFoundError("Run notebook from Rendu/ or Rendu/notebooks/")

MODEL_PATH = ROOT_DIR / "weights" / f"{THEME}_distilroberta" / "hf_model"
OUTPUT_DIR = ROOT_DIR / "outputs" / THEME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

comments = [
    "This update fixed many issues and the app feels much smoother now.",
    "I regret buying this, it crashes all the time and support never answered.",
    "Great value for money, simple to use, and surprisingly fast.",
    "The interface is confusing and the latest patch introduced new bugs.",
    "Average experience: some useful features, but performance is inconsistent.",
    "Excellent customer service, they solved my problem in less than one day.",
    "This game is unplayable after the last update.",
    "Not perfect, but still enjoyable for short sessions.",
    "I absolutely love this product, highly recommended.",
    "The quality is poor and it stopped working after two weeks.",
    "Service was okay, nothing special, but not terrible either.",
    "Fast delivery, solid build quality, and very good overall experience.",
]

print("MODEL_PATH:", MODEL_PATH)
print("Comments:", len(comments))


In [ ]:
URL_RE = re.compile(r"http\S+|www\S+|https\S+", flags=re.MULTILINE)
MENTION_RE = re.compile(r"@\w+")
MULTI_SPACE_RE = re.compile(r"\s+")

def clean_text_for_model(text: str) -> str:
    text = (text or "").strip()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = MULTI_SPACE_RE.sub(" ", text).strip()
    return text

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_PATH))

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device)
model.eval()
print("Device:", device)


In [ ]:
cleaned = [clean_text_for_model(c) for c in comments]

enc = tokenizer(
    cleaned,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    pred = np.argmax(probs, axis=-1)
    proba_pos = probs[:, 1]

df = pd.DataFrame({
    "id": list(range(1, len(comments) + 1)),
    "comment": comments,
    "cleaned_comment": cleaned,
    "pred_label_binary": pred,
    "pred_label_text": ["positive" if p == 1 else "negative" for p in pred],
    "proba_positive": proba_pos,
})

df = df.sort_values("proba_positive", ascending=False).reset_index(drop=True)
df


In [ ]:
csv_path = OUTPUT_DIR / "custom_comments_predictions.csv"
json_path = OUTPUT_DIR / "custom_comments_predictions.json"

df.to_csv(csv_path, index=False)
json_path.write_text(json.dumps(df.to_dict(orient="records"), ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:")
print("-", csv_path)
print("-", json_path)
